While delayed, really need a test environment to test the functionality before pushing. Testing various functions here in this Notebook

In [ ]:
from datetime import datetime, timedelta
from pprint import pprint
from typing import Dict, List
from zoneinfo import ZoneInfo
import docker

client = docker.from_env()

In [ ]:
p2f_api_images = client.images.list("p2f-api*")

In [ ]:
utc = ZoneInfo("UTC")
epoch = datetime(1970, 1, 1, tzinfo=utc)
for im in p2f_api_images:
    print(im.id)
    pprint(im.labels)
    pprint(im.attrs)
    newest_creation_timestamp = epoch
    all_tags = []
    for d in im.history():
        created_time = epoch + timedelta(seconds=d["Created"])
        if created_time > newest_creation_timestamp:
            newest_creation_timestamp = created_time
        if d["Tags"] is not None:
            all_tags += d["Tags"]
    pprint(newest_creation_timestamp)
    pprint(all_tags)

In [ ]:

def highest_epoch_seconds(history: List[Dict]) -> int:
    rv = 0
    for h in history:
        if h["Created"] > rv:
            rv = h["Created"]
    return rv

In [ ]:
p2f_api_images = client.images.list("p2f-api*")
p2f_api_image_ages = {highest_epoch_seconds(x.history()): x.id.split(":")[-1]  for x in p2f_api_images}
print(p2f_api_image_ages)
print(p2f_api_image_ages[max(p2f_api_image_ages.keys())])
p2f_api_image = client.images.get(p2f_api_image_ages[max(p2f_api_image_ages.keys())])
print(p2f_api_image.id)